In [1]:
# [KEEP] - Installs necessary libraries
!pip install -q coqui-tts librosa soundfile matplotlib
import TTS
print("TTS version:", TTS.__version__)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.1 MB/s eta 0:00:00
TTS version: 0.27.5


In [10]:
!apt-get -qq install ffmpeg

In [2]:
!mkdir -p /content/dataset

In [3]:
import os
import torch
import librosa
import soundfile as sf
from TTS.api import TTS
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2",
    gpu=(device == "cuda")
)

/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:22<00:00, 84.3MiB/s]
4.37kiB [00:00, 7.37MiB/s]
361kiB [00:00, 46.5MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 79.0kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 75.3MiB/s]


In [11]:
import os

INPUT_PATH = "/content/dataset"
OUTPUT_PATH = "/content/dataset_wav"

os.makedirs(OUTPUT_PATH, exist_ok=True)

files = [f for f in os.listdir(INPUT_PATH) if f.endswith(".m4a")]

for f in files:
    input_file = os.path.join(INPUT_PATH, f)
    output_file = os.path.join(OUTPUT_PATH, f.replace(".m4a", ".wav"))

    os.system(f"ffmpeg -i '{input_file}' -ar 22050 -ac 1 '{output_file}'")

print("Conversion complete!")

Conversion complete!


In [12]:
def preprocess_text(text):
    if "<excited>" in text:
        return "!!! " + text.replace("<excited>", "")
    elif "<calm>" in text:
        return "... " + text.replace("<calm>", "")
    elif "<whisper>" in text:
        return "... " + text.replace("<whisper>", "")
    return text

In [13]:
def generate_audio(text, speaker_wav, output_file):
    text = preprocess_text(text)

    tts.tts_to_file(
        text=text,
        speaker_wav=speaker_wav,
        language="en",
        file_path=output_file
    )

    display(Audio(output_file))

In [16]:
import os
# Check what files are actually available in the converted folder
converted_files = os.listdir('/content/dataset_wav')
print("Available WAV files:", converted_files)

# Try using one of the existing files, for example 'calm_1.wav'
ref_audio = os.path.join('/content/dataset_wav', 'calm_1.wav')

if os.path.exists(ref_audio):
    generate_audio(
        "This is a basic test of my cloned voice using an available sample.",
        ref_audio,
        "output_normal.wav"
    )
else:
    print(f"Could not find {ref_audio}. Please check the available files list above.")

Available WAV files: ['excited_2.wav', 'hinglish_1.wav', 'calm_2.wav', 'calm_3.wav', 'calm_1.wav', 'hinglish_2.wav', 'excited_3.wav', 'excite_1.wav', 'hinglish_3.wav', 'whisper_1.wav', 'whisper_3.wav', 'whisper_2.wav']


In [23]:
experiments = [
    # (text, corresponding audio file)
    ("Let’s take a moment and think about this carefully before deciding.", "calm_1.wav"),
    ("This is actually insane, I can’t believe this is working!", "excited_1.wav"),
    ("Something about this doesn’t feel completely right to me.", "calm_2.wav"),
    ("Bro honestly this is actually pretty cool yaar, didn’t expect this.", "hinglish_1.wav")
]

for i, (text, file) in enumerate(experiments):
    speaker = os.path.join(OUTPUT_PATH, file)

    if os.path.exists(speaker):
        generate_audio(text, speaker, f"emotion_voice_{i}.wav")
    else:
        print(f"Skipping: {file} not found in {OUTPUT_PATH}")

Skipping: excited_1.wav not found in /content/dataset_wav


In [20]:
texts = [
    "<excited> this is amazing!",
    "<calm> let me think about this...",
    "Bro this is actually crazy yaar"
]

for i, text in enumerate(texts):
    # Using OUTPUT_PATH ('/content/dataset_wav') and an existing file 'calm_1.wav'
    generate_audio(
        text,
        os.path.join(OUTPUT_PATH, "calm_1.wav"),
        f"text_emotion_{i}.wav"
    )

In [24]:
# Updated to use the correct OUTPUT_PATH and verified filenames
speaker_files = [
    "calm_1.wav",
    "excited_2.wav",
    "calm_3.wav",
    "hinglish_1.wav"
]

for i, file in enumerate(speaker_files):
    speaker = os.path.join(OUTPUT_PATH, file)
    if os.path.exists(speaker):
        generate_audio(
            "Testing how tone changes with different voice samples.",
            speaker,
            f"speaker_compare_{i}.wav"
        )
    else:
        print(f"Skipping: {file} not found in {OUTPUT_PATH}")

In [25]:
from google.colab import files

outputs = [f for f in os.listdir() if f.endswith(".wav")]

for f in outputs:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>